# Ablation-only notebook

---
# Colab setup

In [ ]:
# ============================================================
# COLAB SETUP: Google Drive mount + dataset download
# ============================================================
# Runtime > Change runtime type > T4 GPU (original run was on a Kaggle T4 -- keep the same GPU)
from google.colab import drive
drive.mount('/content/drive')

import os, glob
DRIVE_ROOT = "/content/drive/MyDrive/microbrainnet_ablation"   # results persist here across sessions
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Datasets via kagglehub. If it asks for credentials: Colab left panel > Secrets (key icon) >
# add KAGGLE_USERNAME and KAGGLE_KEY (from kaggle.com > Settings > Create New Token).
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
except Exception:
    pass
import kagglehub
PV_ROOT = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
PD_ROOT = kagglehub.dataset_download("nirmalsankalana/plantdoc-dataset")
print("PlantVillage:", PV_ROOT)
print("PlantDoc    :", PD_ROOT)

def _find_split_dir(root, split_name, probe="Tomato___Septoria_leaf_spot"):
    """Locate PlantVillage's color/ or segmented/ folder (must contain the class folder)."""
    for dp, dns, _ in os.walk(root):
        if os.path.basename(dp).lower() == split_name and probe in dns:
            return dp
    raise FileNotFoundError(f"'{split_name}' folder not found under {root}")

PV_COLOR     = _find_split_dir(PV_ROOT, "color")
PV_SEGMENTED = _find_split_dir(PV_ROOT, "segmented")
print("color    :", PV_COLOR)
print("segmented:", PV_SEGMENTED)


Mounted at /content/drive
Using Colab cache for faster access to the 'plantvillage-dataset' dataset.


100%|██████████| 896M/896M [00:12<00:00, 75.7MB/s]

Extracting files...


PlantVillage: /kaggle/input/plantvillage-dataset
PlantDoc    : /root/.cache/kagglehub/datasets/nirmalsankalana/plantdoc-dataset/versions/7
color    : /kaggle/input/plantvillage-dataset/plantvillage dataset/color
segmented: /kaggle/input/plantvillage-dataset/plantvillage dataset/segmented


---
# Part 0: Imports & Configuration

In [ ]:
# ============================================================
# PART 0: IMPORTS AND CONFIGURATION
# ============================================================
import os, sys, time, random, warnings, copy, json, glob, math
import numpy as np
import pandas as pd
import matplotlib
# matplotlib.use('Agg')   # COLAB: commented out so plots show inline
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFilter
from collections import Counter, OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as tvm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             confusion_matrix, classification_report, roc_curve, auc)
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

SEED = 42

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

OUTPUT_DIR  = f"{DRIVE_ROOT}/microbrainnet_v7"   # COLAB: was /kaggle/working/microbrainnet_v7
FIGURES_DIR = f"{OUTPUT_DIR}/figures"
MODELS_DIR  = f"{OUTPUT_DIR}/models"
TABLES_DIR  = f"{OUTPUT_DIR}/tables"
for d in [FIGURES_DIR, MODELS_DIR, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

# ------------------------------------------------------------------
# FIX #8 -- the v6 student used 42,607 params = 41.6 KB INT8, i.e. 8% of the
# stated <500 KB budget. 96px throws away the fine texture that separates
# Tomato Septoria (many small spots) from Potato Early Blight (concentric rings).
# 128px + wider channels lands around 230 KB INT8 -- still comfortably inside budget,
# and ~250-400 ms/inference on an ESP32-S3 with PSRAM.
# ------------------------------------------------------------------
IMG_SIZE          = 128
TEACHER_IMG_SIZE  = 224
NUM_CLASSES       = 3
STUDENT_CHANNELS  = (32, 48, 64, 96, 96)     # v6 was (16, 24, 32, 48, 48)

BATCH_SIZE        = 48    # source
TARGET_BATCH_SIZE = 32    # FIX #7: smaller batch + drop_last=False keeps all 364 target images
TEACHER_BATCH_SIZE= 24

CLASSES = ["Tomato_Septoria_Leaf_Spot", "Potato_Early_Blight", "Corn_Leaf_Blight"]

# ---------------- stage switches ----------------
RUN_SANITY_GATE        = True    # Part 0B: source-only run. DO NOT SKIP on the first execution.
USE_BG_RANDOMIZATION   = True    # FIX #6
USE_TEACHER_KD         = True
TEACHER_USE_PSEUDO     = False   # FIX #1: OFF. This is what poisoned v6. Turn on only after
                                 #         the teacher's clean baseline is known.
STUDENT_USE_PSEUDO     = True    # gated + auto-disabling (see Part 3/4)
RUN_MULTI_SEED         = True    # Part 7 (expensive: re-runs the student per seed) -- turned ON:
                                  # a single-seed number is not defensible on a 79-image test set
SEEDS                  = [42, 1337, 2024]

# ---------------- teacher ----------------
TEACHER_BACKBONE   = 'mobilenet_v3_large'
TEACHER_EPOCHS     = 80
TEACHER_LR_BACKBONE= 3e-4        # FIX #10: discriminative LR instead of a cosmetic block freeze
TEACHER_LR_HEAD    = 3e-3
TEACHER_WARMUP     = 3
TEACHER_PATIENCE   = 30

# ---------------- student ----------------
EPOCHS         = 120
LR             = 1.5e-3
WEIGHT_DECAY   = 1e-4
LABEL_SMOOTHING= 0.1
WARMUP_EPOCHS  = 5
PATIENCE       = 40

# ---------------- domain adaptation ----------------
# EXPERIMENT: IM-only. v7 ran four alignment losses together (CORAL + MMD + IM + MCC).
# This run switches all four off except Information Maximization to see how much of the
# adaptation the plain IM term can carry on its own -- CORAL/MMD/MCC stay fully implemented
# below (Part 3) and can be flipped back on by setting these four switches to True.
USE_CORAL = False
USE_MMD   = False
USE_IM    = True
USE_MCC   = False

# FIX #9: CORAL down from 5.0. In v6 CORAL fell to 1e-4 while target accuracy never moved --
# the classic signature of degenerate alignment (domains matched by destroying class structure).
LAMBDA_CORAL_MAX  = 0.5
LAMBDA_MMD_MAX    = 0.5
LAMBDA_IM_MAX     = 0.5    # FIX #3: Information Maximization replaces bare entropy-min
LAMBDA_MCC_MAX    = 1.0    # FIX #3: Minimum Class Confusion -- class-aware, stable
LAMBDA_PSEUDO_MAX = 0.3

# FIX #2: v6 iterated the source loader (41 batches) and cycled the 5-batch target loader
# 8.2x per epoch, so 364 target images dominated every alignment term. Cap it.
DA_TARGET_PASSES  = 2

# ---------------- pseudo-labeling (FIX #1) ----------------
PSEUDO_CONF            = 0.95
PSEUDO_TOPK_PER_CLASS  = 8      # class-balanced instead of a global threshold
PSEUDO_MIN_MONITOR_ACC = 0.85   # hard safety valve: below this, self-training switches itself off
PSEUDO_PLATEAU_EPOCHS  = 8      # only start once target-val has stopped improving
EMA_DECAY              = 0.999

# ---------------- knowledge distillation (FIX #4) ----------------
LAMBDA_KD      = 1.0    # constant, NOT ramped, and no longer 3x the CE weight
LAMBDA_FEATKD  = 1.0
KD_TEMPERATURE = 4.0

SMOOTH_WINDOW = 5

CFG = dict(coral=LAMBDA_CORAL_MAX, mmd=LAMBDA_MMD_MAX, im=LAMBDA_IM_MAX, mcc=LAMBDA_MCC_MAX,
           pseudo=LAMBDA_PSEUDO_MAX, kd=LAMBDA_KD, featkd=LAMBDA_FEATKD, T=KD_TEMPERATURE,
           da_passes=DA_TARGET_PASSES, pseudo_conf=PSEUDO_CONF,
           pseudo_topk=PSEUDO_TOPK_PER_CLASS, pseudo_min_acc=PSEUDO_MIN_MONITOR_ACC)

print(f"\nIMG={IMG_SIZE} channels={STUDENT_CHANNELS} BS={BATCH_SIZE}/{TARGET_BATCH_SIZE}")
print(f"DA technique switches: CORAL={USE_CORAL} MMD={USE_MMD} IM={USE_IM} MCC={USE_MCC}")
print(f"DA lambdas (only active ones apply): coral={LAMBDA_CORAL_MAX} mmd={LAMBDA_MMD_MAX} "
      f"IM={LAMBDA_IM_MAX} MCC={LAMBDA_MCC_MAX}")
print(f"KD: lambda={LAMBDA_KD} (constant) T={KD_TEMPERATURE} feat={LAMBDA_FEATKD}")
print(f"Pseudo: teacher={TEACHER_USE_PSEUDO} student={STUDENT_USE_PSEUDO} "
      f"conf>={PSEUDO_CONF} topk={PSEUDO_TOPK_PER_CLASS}/class auto-off<{PSEUDO_MIN_MONITOR_ACC}")

Device: cuda | PyTorch: 2.11.0+cu130
GPU: Tesla T4

IMG=128 channels=(32, 48, 64, 96, 96) BS=48/32
DA technique switches: CORAL=False MMD=False IM=True MCC=False
DA lambdas (only active ones apply): coral=0.5 mmd=0.5 IM=0.5 MCC=1.0
KD: lambda=1.0 (constant) T=4.0 feat=1.0
Pseudo: teacher=False student=True conf>=0.95 topk=8/class auto-off<0.85


---
# Part 1: Data — indexing, splits, domain-gap augmentation

In [ ]:
# ============================================================
# PART 1A: DATASET INDEX
# ============================================================
SOURCE_ALL    = PV_COLOR       # COLAB (was /kaggle/input/.../plantvillage-dataset/color)
SEGMENTED_ALL = PV_SEGMENTED   # COLAB (was /kaggle/input/.../plantvillage-dataset/segmented)
TARGET_ALL    = PD_ROOT        # COLAB (was /kaggle/input/datasets/nirmalsankalana/plantdoc-dataset)

SOURCE_CLASS_DIRS = {
    "Tomato_Septoria_Leaf_Spot": ["Tomato___Septoria_leaf_spot"],
    "Potato_Early_Blight":       ["Potato___Early_blight"],
    "Corn_Leaf_Blight":          ["Corn_(maize)___Northern_Leaf_Blight"],
}
TARGET_CLASS_DIRS = {
    "Tomato_Septoria_Leaf_Spot": ["Tomato Septoria leaf spot"],
    "Potato_Early_Blight":       ["Potato leaf early blight"],
    "Corn_Leaf_Blight":          ["Corn leaf blight"],
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


class LeafIndex:
    """Path/label index only -- no transforms. Walks root_dir recursively and pools
    every image under any folder whose normalized name matches an alias for a class."""

    def __init__(self, root_dir, classes, class_dir_map, domain_label=0, name=""):
        self.img_paths, self.labels = [], []
        self.classes, self.domain_label = classes, domain_label

        def _norm(s):
            return s.strip().lower().replace(" ", "").replace("_", "")

        for idx, cls in enumerate(classes):
            alias = {_norm(a) for a in class_dir_map.get(cls, [cls])}
            matched = [dp for dp, _, _ in os.walk(root_dir) if _norm(os.path.basename(dp)) in alias]
            for d in matched:
                for f in sorted(os.listdir(d)):
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                        self.img_paths.append(os.path.join(d, f)); self.labels.append(idx)
            if not matched:
                print(f"    WARNING: no folder matched '{cls}' under {root_dir}")

        print(f"  [{name}] {len(self.img_paths)} images")
        for i, c in enumerate(classes):
            print(f"    {c}: {self.labels.count(i)}")

    def subset(self, indices):
        s = copy.copy(self)
        s.img_paths = [self.img_paths[i] for i in indices]
        s.labels    = [self.labels[i] for i in indices]
        return s


def stratified_split_indices(labels, val_frac=0.15, test_frac=0.15, seed=SEED):
    idx_all = list(range(len(labels)))
    idx_tv, idx_test = train_test_split(idx_all, test_size=test_frac, stratify=labels,
                                        random_state=seed)
    lab_tv = [labels[i] for i in idx_tv]
    idx_tr, idx_val = train_test_split(idx_tv, test_size=val_frac / (1 - test_frac),
                                       stratify=lab_tv, random_state=seed)
    return idx_tr, idx_val, idx_test


for nm, p in [('PlantVillage color', SOURCE_ALL), ('PlantVillage segmented', SEGMENTED_ALL),
              ('PlantDoc', TARGET_ALL)]:
    print(f"  {nm:24s}: {'OK ' if os.path.isdir(p) else '!! NOT FOUND !!'} {p}")

print("\n--- Source (PlantVillage) ---")
src_index = LeafIndex(SOURCE_ALL, CLASSES, SOURCE_CLASS_DIRS, 0, "Source")
print("\n--- Target (PlantDoc) ---")
tgt_index = LeafIndex(TARGET_ALL, CLASSES, TARGET_CLASS_DIRS, 1, "Target")

src_tr_i, src_va_i, src_te_i = stratified_split_indices(src_index.labels)
tgt_tr_i, tgt_va_i, tgt_te_i = stratified_split_indices(tgt_index.labels)

src_train_idx = src_index.subset(src_tr_i)
src_val_idx   = src_index.subset(src_va_i)
src_test_idx  = src_index.subset(src_te_i)
tgt_train_idx = tgt_index.subset(tgt_tr_i)
tgt_val_idx   = tgt_index.subset(tgt_va_i)
tgt_test_idx  = tgt_index.subset(tgt_te_i)

print(f"\nSource  train/val/test: {len(src_train_idx.labels)}/"
      f"{len(src_val_idx.labels)}/{len(src_test_idx.labels)}")
print(f"Target  train/val/test: {len(tgt_train_idx.labels)}/"
      f"{len(tgt_val_idx.labels)}/{len(tgt_test_idx.labels)}")
print(f"\nNOTE: target test = {len(tgt_test_idx.labels)} images -> 1 image = "
      f"{100/len(tgt_test_idx.labels):.2f} accuracy points. Never quote a single point estimate.")

  PlantVillage color      : OK  /kaggle/input/plantvillage-dataset/plantvillage dataset/color
  PlantVillage segmented  : OK  /kaggle/input/plantvillage-dataset/plantvillage dataset/segmented
  PlantDoc                : OK  /root/.cache/kagglehub/datasets/nirmalsankalana/plantdoc-dataset/versions/7

--- Source (PlantVillage) ---
  [Source] 3756 images
    Tomato_Septoria_Leaf_Spot: 1771
    Potato_Early_Blight: 1000
    Corn_Leaf_Blight: 985

--- Target (PlantDoc) ---
  [Target] 522 images
    Tomato_Septoria_Leaf_Spot: 157
    Potato_Early_Blight: 171
    Corn_Leaf_Blight: 194

Source  train/val/test: 2628/564/564
Target  train/val/test: 364/79/79

NOTE: target test = 79 images -> 1 image = 1.27 accuracy points. Never quote a single point estimate.


In [ ]:
# ============================================================
# PART 1B: BACKGROUND RANDOMIZATION  (FIX #6)
# ============================================================
# PlantVillage: centred leaf, uniform grey backdrop. PlantDoc: leaf in a cluttered field.
# A source-only model latches onto the backdrop as a shortcut. Cutting the leaf out and
# pasting it onto real field backgrounds removes that shortcut at the data level, which is
# more effective than any loss-side alignment term.
#
# Mask source, in order of preference:
#   1. PlantVillage's own segmented/ split (exact, free) -- matched by filename stem
#   2. Border-colour heuristic + morphological cleanup (works because the backdrop is uniform)
# Backgrounds: random crops of PlantDoc TRAIN images only (never val/test -> no leakage).

from scipy import ndimage as ndi


def build_segmented_lookup(seg_root, class_dir_map, classes):
    """stem -> segmented path. PlantVillage segmented filenames carry a '_final_masked'
    suffix, so match on the leading UUID-ish stem rather than the full filename."""
    lut = {}
    if not (seg_root and os.path.isdir(seg_root)):
        return lut

    def _norm(s):
        return s.strip().lower().replace(" ", "").replace("_", "")

    alias = set()
    for c in classes:
        alias |= {_norm(a) for a in class_dir_map.get(c, [c])}
    for dp, _, files in os.walk(seg_root):
        if _norm(os.path.basename(dp)) not in alias:
            continue
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                stem = os.path.splitext(f)[0].replace('_final_masked', '')
                lut[stem] = os.path.join(dp, f)
    return lut


SEG_LUT = build_segmented_lookup(SEGMENTED_ALL, SOURCE_CLASS_DIRS, CLASSES) if USE_BG_RANDOMIZATION else {}
print(f"Segmented-mask lookup: {len(SEG_LUT)} entries "
      f"({'exact masks available' if SEG_LUT else 'falling back to the border heuristic'})")


def estimate_leaf_mask(img, seg_path=None):
    """Boolean HxW mask, True = leaf. Returns None when the result looks untrustworthy,
    in which case the caller simply skips background randomization for that image."""
    if seg_path is not None and os.path.exists(seg_path):
        try:
            seg = np.asarray(Image.open(seg_path).convert("RGB").resize(img.size), np.float32)
            m = seg.sum(2) > 24          # segmented split paints the background pure black
            if 0.05 < m.mean() < 0.97:
                return ndi.binary_closing(m, np.ones((5, 5)))
        except Exception:
            pass

    a = np.asarray(img.convert("RGB"), np.float32)
    b = np.concatenate([a[:6].reshape(-1, 3), a[-6:].reshape(-1, 3),
                        a[:, :6].reshape(-1, 3), a[:, -6:].reshape(-1, 3)])
    med = np.median(b, 0)
    d = np.linalg.norm(a - med, axis=2)
    m = d > max(35.0, np.percentile(d, 40))
    m = ndi.binary_opening(m, np.ones((3, 3)))
    m = ndi.binary_closing(m, np.ones((9, 9)))
    m = ndi.binary_fill_holes(m)

    lab, n = ndi.label(m)                # keep only the largest blob (the leaf)
    if n > 1:
        sizes = ndi.sum(m, lab, range(1, n + 1))
        m = lab == (int(np.argmax(sizes)) + 1)
    if not (0.05 < m.mean() < 0.95):
        return None
    return m


def build_background_bank(index_obj, n=180, size=320):
    """Field backgrounds taken from PlantDoc TRAIN images only."""
    rng = np.random.RandomState(SEED)
    paths = list(index_obj.img_paths)
    rng.shuffle(paths)
    bank = []
    for p in paths[:n]:
        try:
            bank.append(Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR))
        except Exception:
            continue
    return bank


BG_BANK = build_background_bank(tgt_train_idx) if USE_BG_RANDOMIZATION else []
print(f"Background bank: {len(BG_BANK)} field images (PlantDoc TRAIN split only)")


MASK_SIZE = 192


def precompute_masks(index_obj):
    """Masks are computed ONCE here and stored bit-packed (~6 KB each), instead of being
    recomputed inside the dataloader on every __getitem__. Without this the scipy
    morphology becomes the training bottleneck."""
    if not USE_BG_RANDOMIZATION:
        return {}
    cache, t0 = {}, time.time()
    for n, p in enumerate(index_obj.img_paths):
        try:
            im = Image.open(p).convert("RGB").resize((MASK_SIZE, MASK_SIZE), Image.BILINEAR)
            stem = os.path.splitext(os.path.basename(p))[0]
            m = estimate_leaf_mask(im, SEG_LUT.get(stem))
            if m is not None:
                cache[p] = np.packbits(m)
        except Exception:
            pass
        if (n + 1) % 500 == 0:
            print(f"    masks {n+1}/{len(index_obj.img_paths)} ({time.time()-t0:.0f}s)")
    print(f"  usable leaf masks: {len(cache)}/{len(index_obj.img_paths)} "
          f"in {time.time()-t0:.0f}s")
    return cache


MASK_CACHE = precompute_masks(src_train_idx) if USE_BG_RANDOMIZATION else {}


def randomize_background(img, path, rng):
    """Composite the leaf onto a random field background with a soft alpha edge."""
    if not BG_BANK or path not in MASK_CACHE:
        return img
    m = np.unpackbits(MASK_CACHE[path])[:MASK_SIZE * MASK_SIZE].reshape(MASK_SIZE, MASK_SIZE)

    bg = BG_BANK[rng.randint(len(BG_BANK))]
    W, H = img.size
    bw, bh = bg.size
    cw = rng.randint(int(bw * 0.4), bw + 1)
    x0 = rng.randint(0, bw - cw + 1); y0 = rng.randint(0, bh - cw + 1)
    bg = bg.crop((x0, y0, x0 + cw, y0 + cw)).resize((W, H), Image.BILINEAR)

    alpha = (Image.fromarray((m * 255).astype(np.uint8))
             .resize((W, H), Image.BILINEAR).filter(ImageFilter.GaussianBlur(1.5)))
    out = Image.composite(img, bg, alpha)
    if rng.rand() < 0.3:                 # sometimes blur the background only
        out = Image.composite(out, bg.filter(ImageFilter.GaussianBlur(2.0)), alpha)
    return out

Segmented-mask lookup: 3756 entries (exact masks available)
Background bank: 180 field images (PlantDoc TRAIN split only)
    masks 500/2628 (14s)
    masks 1000/2628 (28s)
    masks 1500/2628 (42s)
    masks 2000/2628 (55s)
    masks 2500/2628 (69s)
  usable leaf masks: 2628/2628 in 72s


In [ ]:
# ============================================================
# PART 1C: DualResDataset  (FIX #4 + FIX #6)
# ============================================================
# One augmentation, rendered at both resolutions. This is the cell that repairs distillation:
# v6 augmented at 96px and bilinearly upsampled to 224px for the teacher, so the teacher saw
# blurry stretched inputs it was never fine-tuned on and emitted soft targets that were then
# weighted 3x the CE loss. Here both views come from the same 256px augmented image.
#
# Returns (x_student, x_teacher, x_weak_student, x_weak_teacher, label, domain).
# Two weak views are needed because pseudo-labels must be generated at whatever resolution the
# model being trained consumes -- 128px for the student, 224px for the teacher.
# Unrequested views are 1-element placeholders so the default collate still stacks cleanly.

_PLACEHOLDER = torch.zeros(1)


class DualResDataset(Dataset):
    def __init__(self, index_obj, train=True, emit_teacher=False, emit_weak=False,
                 bg_random_p=0.0, img_size=IMG_SIZE, teacher_size=TEACHER_IMG_SIZE):
        self.idx = index_obj
        self.train = train
        self.emit_teacher = emit_teacher
        self.emit_weak = emit_weak
        self.bg_p = bg_random_p if index_obj.domain_label == 0 else 0.0
        self.img_size, self.teacher_size = img_size, teacher_size

        # -- strong, domain-gap-aware augmentation (FIX #6) --
        # RandomResizedCrop(0.25-1.0) is the important one: it forces the model onto lesion
        # texture instead of whole-leaf silhouette, which does not survive the domain shift.
        self.geo_train = transforms.Compose([
            transforms.RandomResizedCrop(256, scale=(0.25, 1.0), ratio=(0.75, 1.33)),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.2),
            transforms.RandomApply([transforms.RandomRotation(25)], p=0.5),
            transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            transforms.RandomGrayscale(p=0.1),
            transforms.RandomApply([transforms.GaussianBlur(5, (0.1, 2.0))], p=0.3),
        ])
        self.geo_eval = transforms.Resize((256, 256))
        self.geo_weak = transforms.Compose([transforms.Resize((256, 256)),
                                            transforms.RandomHorizontalFlip(0.5)])

        self.to_student = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3)])
        self.to_teacher = transforms.Compose([
            transforms.Resize((teacher_size, teacher_size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
        self.erase = transforms.RandomErasing(p=0.25, scale=(0.02, 0.15), value='random')

    def __len__(self):
        return len(self.idx.img_paths)

    def __getitem__(self, i):
        path = self.idx.img_paths[i]
        img = Image.open(path).convert("RGB")
        y, d = self.idx.labels[i], self.idx.domain_label

        if self.train and self.bg_p > 0:
            rng = np.random.RandomState((i * 7919 + int(time.time() * 1e3)) % (2 ** 31))
            if rng.rand() < self.bg_p:
                img = randomize_background(img.resize((256, 256), Image.BILINEAR), path, rng)

        base = self.geo_train(img) if self.train else self.geo_eval(img)

        xs = self.to_student(base)
        if self.train:
            xs = self.erase(xs)
        xt = self.to_teacher(base) if self.emit_teacher else _PLACEHOLDER
        if self.emit_weak:
            wbase = self.geo_weak(img)
            xw_s = self.to_student(wbase)
            xw_t = self.to_teacher(wbase) if self.emit_teacher else _PLACEHOLDER
        else:
            xw_s = xw_t = _PLACEHOLDER
        return xs, xt, xw_s, xw_t, y, d

In [ ]:
# ============================================================
# PART 1D: DATASETS AND LOADERS
# ============================================================
NW = 2          # COLAB: 2 CPU cores. Keep this SAME value in every session.

def mk_loader(ds, bs, shuffle, drop_last=False):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=NW,
                      pin_memory=True, drop_last=drop_last, persistent_workers=(NW > 0))

BG_P = 0.5 if USE_BG_RANDOMIZATION else 0.0

# -- train --
src_train_ds = DualResDataset(src_train_idx, True,  emit_teacher=True, bg_random_p=BG_P)
tgt_train_ds = DualResDataset(tgt_train_idx, True,  emit_teacher=True, emit_weak=True)

# -- eval (no augmentation anywhere) --
src_val_ds   = DualResDataset(src_val_idx,   False, emit_teacher=True)
src_test_ds  = DualResDataset(src_test_idx,  False, emit_teacher=True)
tgt_val_ds   = DualResDataset(tgt_val_idx,   False, emit_teacher=True)
tgt_test_ds  = DualResDataset(tgt_test_idx,  False, emit_teacher=True)

# FIX #12: v6 measured "source train accuracy" through the augmented training loader, which
# understated it and therefore corrupted the reported generalization gap. Clean loader:
src_train_eval_ds = DualResDataset(src_train_idx, False, emit_teacher=True)

src_train_loader = mk_loader(src_train_ds, BATCH_SIZE, True,  drop_last=True)
# FIX #7: drop_last=False -- v6 silently discarded 44 of 364 target images every epoch.
tgt_train_loader = mk_loader(tgt_train_ds, TARGET_BATCH_SIZE, True, drop_last=False)

src_val_loader        = mk_loader(src_val_ds,   BATCH_SIZE, False)
src_test_loader       = mk_loader(src_test_ds,  BATCH_SIZE, False)
tgt_val_loader        = mk_loader(tgt_val_ds,   BATCH_SIZE, False)
tgt_test_loader       = mk_loader(tgt_test_ds,  BATCH_SIZE, False)
src_train_eval_loader = mk_loader(src_train_eval_ds, BATCH_SIZE, False)

print(f"source batches/epoch : {len(src_train_loader)}")
print(f"target batches/epoch : {len(tgt_train_loader)}")
print(f"DA steps/epoch       : {min(len(src_train_loader), DA_TARGET_PASSES*len(tgt_train_loader))}"
      f"  -> each target image seen ~{DA_TARGET_PASSES}x/epoch")
print(f"  (v6 cycled the target loader {len(src_train_loader)/max(len(tgt_train_loader),1):.1f}x "
      f"per epoch -- that over-weighting is what drove CORAL to 1e-4 with no accuracy gain)")

xb = next(iter(src_train_loader))
print(f"\nbatch shapes: student={tuple(xb[0].shape)} teacher={tuple(xb[1].shape)}")
print(f"student range: [{xb[0].min():.2f}, {xb[0].max():.2f}]  (expect ~[-1, 1])")
xt_b = next(iter(tgt_train_loader))
print(f"target weak views: student={tuple(xt_b[2].shape)} teacher={tuple(xt_b[3].shape)}")

source batches/epoch : 54
target batches/epoch : 12
DA steps/epoch       : 24  -> each target image seen ~2x/epoch
  (v6 cycled the target loader 4.5x per epoch -- that over-weighting is what drove CORAL to 1e-4 with no accuracy gain)

batch shapes: student=(48, 3, 128, 128) teacher=(48, 3, 224, 224)
student range: [-3.86, 4.09]  (expect ~[-1, 1])
target weak views: student=(32, 3, 128, 128) teacher=(32, 3, 224, 224)


---
# Part 2: Architecture

In [ ]:
# ============================================================
# PART 2: MicroBrainNet-DA v7
# ============================================================

class InvertedResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, expand_ratio=2):
        super().__init__()
        self.use_residual = (stride == 1 and in_ch == out_ch)
        hid = in_ch * expand_ratio
        layers = []
        if expand_ratio != 1:
            layers += [nn.Conv2d(in_ch, hid, 1, bias=False), nn.BatchNorm2d(hid),
                       nn.ReLU6(inplace=True)]
        layers += [nn.Conv2d(hid, hid, 3, stride, 1, groups=hid, bias=False),
                   nn.BatchNorm2d(hid), nn.ReLU6(inplace=True),
                   nn.Conv2d(hid, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch)]
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.conv(x) if self.use_residual else self.conv(x)


class SqueezeExciteBlock(nn.Module):
    def __init__(self, ch, reduction=4):
        super().__init__()
        mid = max(ch // reduction, 8)
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excite = nn.Sequential(nn.Linear(ch, mid), nn.ReLU(inplace=True),
                                    nn.Linear(mid, ch), nn.Sigmoid())

    def forward(self, x):
        b, c, _, _ = x.shape
        return x * self.excite(self.squeeze(x).view(b, c)).view(b, c, 1, 1)


class MicroViTBlock(nn.Module):
    """Linear attention, O(n). ReLU6 in the FFN (not GELU) so every op has a TFLite Micro kernel."""

    def __init__(self, dim, num_tokens, ffn_ratio=2):
        super().__init__()
        self.q_proj = nn.Linear(dim, dim); self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim); self.out_proj = nn.Linear(dim, dim)
        self.norm1 = nn.LayerNorm(dim); self.norm2 = nn.LayerNorm(dim)
        # learned positional embedding -- v6 had none, so patch layout was invisible
        self.pos = nn.Parameter(torch.zeros(1, num_tokens, dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        h = dim * ffn_ratio
        self.ffn = nn.Sequential(nn.Linear(dim, h), nn.ReLU6(inplace=True), nn.Dropout(0.1),
                                 nn.Linear(h, dim), nn.Dropout(0.1))

    def linear_attention(self, q, k, v):
        q = F.elu(q) + 1
        k = F.elu(k) + 1
        kv = torch.bmm(k.transpose(1, 2), v)
        num = torch.bmm(q, kv)
        den = torch.bmm(q, k.sum(1, keepdim=True).transpose(1, 2)).clamp(min=1e-4)
        return num / den

    def forward(self, x):
        x = x + self.pos
        xn = self.norm1(x)
        x = x + self.out_proj(self.linear_attention(self.q_proj(xn), self.k_proj(xn),
                                                    self.v_proj(xn)))
        return x + self.ffn(self.norm2(x))


class MicroBrainNetDA(nn.Module):
    """128x128x3 RGB -> 3 logits. Downsampling: stem s2, b1 s2, b2 s2, b3 s2, b4 s1
    => 8x8 = 64 tokens for the ViT stage."""

    def __init__(self, num_classes=NUM_CLASSES, channels=STUDENT_CHANNELS, img_size=IMG_SIZE,
                 use_se=True, use_vit=True):
        super().__init__()
        c0, c1, c2, c3, c4 = channels
        self.use_se, self.use_vit = use_se, use_vit
        self.stem = nn.Sequential(nn.Conv2d(3, c0, 3, 2, 1, bias=False),
                                  nn.BatchNorm2d(c0), nn.ReLU6(inplace=True))
        self.block1 = InvertedResidualBlock(c0, c1, 2, 2)
        self.block2 = InvertedResidualBlock(c1, c2, 2, 2)
        self.block3 = InvertedResidualBlock(c2, c3, 2, 2)
        self.block4 = InvertedResidualBlock(c3, c4, 1, 2)
        self.se = SqueezeExciteBlock(c4) if use_se else nn.Identity()
        self.grid = img_size // 16
        self.vit = MicroViTBlock(c4, self.grid * self.grid) if use_vit else nn.Identity()
        self.classifier = nn.Sequential(nn.LayerNorm(c4), nn.Dropout(0.2),
                                        nn.Linear(c4, num_classes))
        self.feature_dim = c4
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def extract_features(self, x):
        x = self.stem(x)
        x = self.block1(x); x = self.block2(x); x = self.block3(x); x = self.block4(x)
        x = self.se(x)
        if self.use_vit:
            x = x.flatten(2).transpose(1, 2)
            x = self.vit(x)
            return x.mean(1)
        return F.adaptive_avg_pool2d(x, 1).flatten(1)

    def forward(self, x):
        return self.classifier(self.extract_features(x))


class TeacherNet(nn.Module):
    """ImageNet-pretrained backbone behind the same extract_features()/classifier interface,
    so it drops straight into train_da_epoch(). Training-time only -- never quantized."""

    def __init__(self, num_classes=NUM_CLASSES, backbone_name=TEACHER_BACKBONE):
        super().__init__()
        if backbone_name == 'mobilenet_v3_large':
            bb = tvm.mobilenet_v3_large(weights=tvm.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
        elif backbone_name == 'mobilenet_v3_small':
            bb = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        elif backbone_name == 'efficientnet_b0':
            bb = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1)
        else:
            raise ValueError(backbone_name)
        self.features = bb.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.feature_dim = bb.classifier[0].in_features
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(self.feature_dim, num_classes))

    def extract_features(self, x):
        return self.pool(self.features(x)).flatten(1)

    def forward(self, x):
        return self.classifier(self.extract_features(x))


model = MicroBrainNetDA().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Student params : {total_params:,}")
print(f"  FP32 : {total_params*4/1024:.1f} KB")
print(f"  INT8 : {total_params/1024:.1f} KB   (budget 500 KB -> "
      f"{total_params/1024/500*100:.0f}% used; v6 used 8%)")
print(f"  ViT token grid: {model.grid}x{model.grid} = {model.grid**2} tokens")
with torch.no_grad():
    print("forward:", tuple(model(torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)).shape))

Student params : 167,131
  FP32 : 652.9 KB
  INT8 : 163.2 KB   (budget 500 KB -> 33% used; v6 used 8%)
  ViT token grid: 8x8 = 64 tokens
forward: (2, 3)


---
# Part 3: Losses

In [ ]:
# ============================================================
# PART 3: LOSS FUNCTIONS
# ============================================================

def standardize_feats(x, eps=1e-5):
    return (x - x.mean(0, keepdim=True)) / (x.std(0, keepdim=True) + eps)


def compute_covariance(x):
    n = x.size(0)
    ones = torch.ones(1, n, device=x.device)
    mean = ones @ x / n
    return (x.t() @ x - mean.t() @ mean) / max(n - 1, 1)


def coral_loss(fs, ft):
    s, t = standardize_feats(fs), standardize_feats(ft)
    d = s.size(1)
    return (compute_covariance(s) - compute_covariance(t)).pow(2).sum() / (4.0 * d * d)


def gaussian_kernel_matrix(s, t, kernel_mul=2.0, kernel_num=5):
    total = torch.cat([s, t], 0)
    n = total.size(0)
    l2 = ((total.unsqueeze(0) - total.unsqueeze(1)) ** 2).sum(2)
    bw = torch.sum(l2.detach()) / max(n * n - n, 1)
    bw = bw / (kernel_mul ** (kernel_num // 2)) + 1e-8
    return sum(torch.exp(-l2 / (bw * kernel_mul ** i)) for i in range(kernel_num))


def mmd_loss(fs, ft):
    s, t = standardize_feats(fs), standardize_feats(ft)
    ns = s.size(0)
    k = gaussian_kernel_matrix(s, t)
    return k[:ns, :ns].mean() + k[ns:, ns:].mean() - k[:ns, ns:].mean() - k[ns:, :ns].mean()


def im_loss(logits):
    """Information Maximization = confident per sample MINUS diverse across the batch.

    FIX #3. Bare entropy minimization (v6) has a degenerate optimum: predict one class for
    everything. The batch-marginal term makes that the WORST solution instead of the best.
    Returns ent_individual - ent_batch; both terms are minimized together."""
    p = F.softmax(logits, 1)
    ent_ind = -(p * torch.log(p + 1e-8)).sum(1).mean()
    pm = p.mean(0)
    ent_batch = -(pm * torch.log(pm + 1e-8)).sum()
    return ent_ind - ent_batch


def mcc_loss(logits, T=2.5):
    """Minimum Class Confusion (Jin et al.). Class-aware, so unlike CORAL/MMD it cannot be
    satisfied by a feature collapse -- off-diagonal mass only falls if target predictions
    become mutually distinct."""
    p = F.softmax(logits / T, 1)
    w = (1 + (-(p * torch.log(p + 1e-8)).sum(1))).detach()
    w = (w / w.sum()) * p.size(0)
    C = (p * w.unsqueeze(1)).t() @ p
    C = C / (C.sum(1, keepdim=True) + 1e-8)
    return (C.sum() - C.trace()) / logits.size(1)


def kd_loss_fn(s_logits, t_logits, T=4.0):
    return F.kl_div(F.log_softmax(s_logits / T, 1), F.softmax(t_logits / T, 1),
                    reduction='batchmean') * (T * T)


def feat_kd_loss(s_proj, t_feat):
    """Cosine feature distillation. With a ~70x capacity gap, logit-only KD transfers little;
    matching normalized feature direction gives the student a much denser signal."""
    return F.mse_loss(F.normalize(s_proj, dim=1), F.normalize(t_feat.detach(), dim=1))


@torch.no_grad()
def balanced_pseudo(logits, k_per_class=PSEUDO_TOPK_PER_CLASS, min_conf=PSEUDO_CONF,
                    n_classes=NUM_CLASSES):
    """Class-balanced top-k selection (FIX #1). A global threshold lets the dominant class
    supply nearly every pseudo-label, which is how v6 reinforced its own Potato->Tomato error."""
    probs = F.softmax(logits, 1)
    conf, lab = probs.max(1)
    keep = torch.zeros_like(conf, dtype=torch.bool)
    for c in range(n_classes):
        idx = ((lab == c) & (conf >= min_conf)).nonzero(as_tuple=True)[0]
        if idx.numel():
            top = conf[idx].topk(min(k_per_class, idx.numel())).indices
            keep[idx[top]] = True
    return keep, lab


def ramp(epoch, max_epoch, warm=0):
    """Sigmoid ramp for the alignment weights. KD is deliberately NOT ramped (FIX #4):
    ramping it up meant it peaked exactly when it should have been fading out."""
    if epoch <= warm:
        return 0.0
    p = (epoch - warm) / max(max_epoch - warm, 1)
    return float(2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0)


print("Losses ready: coral, mmd, im (IM replaces entropy-min), mcc, kd, feat_kd, balanced_pseudo")

Losses ready: coral, mmd, im (IM replaces entropy-min), mcc, kd, feat_kd, balanced_pseudo


---
# Part 4: Training utilities

In [ ]:
# ============================================================
# PART 4: TRAINING UTILITIES
# ============================================================

class EMA:
    """Exponential moving average of the student, used only to generate pseudo-labels."""

    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.model.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(self.decay).add_(msd[k].detach(), alpha=1 - self.decay)
            else:
                v.copy_(msd[k])


class EarlyStopping:
    def __init__(self, patience=PATIENCE, min_delta=1e-3):
        self.patience, self.min_delta = patience, min_delta
        self.counter, self.best, self.early_stop, self.best_model = 0, None, False, None

    def __call__(self, score, model):
        if self.best is None or score > self.best + self.min_delta:
            self.best, self.counter = score, 0
            self.best_model = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


@torch.no_grad()
def evaluate(model, loader, criterion, device, which='s'):
    """which='s' -> 128px/[-1,1] student view; which='t' -> 224px/ImageNet teacher view."""
    model.eval()
    tot, preds, labs, probs = 0.0, [], [], []
    for xs, xt, xw_s, xw_t, y, d in loader:
        x = (xs if which == 's' else xt).to(device)
        y = y.to(device)
        logits = model(x)
        tot += criterion(logits, y).item() * x.size(0)
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(y.cpu().numpy())
        probs.extend(F.softmax(logits, 1).cpu().numpy())
    n = len(labs)
    return (tot / n, accuracy_score(labs, preds), f1_score(labs, preds, average='weighted'),
            np.array(preds), np.array(labs), np.array(probs))


@torch.no_grad()
def adapt_bn(model, loader, device, which='s', n_pass=3):
    """AdaBN (FIX #5): recompute BN running statistics from TARGET data only.
    momentum=None makes each BN accumulate a cumulative average over the passes.
    Must run before INT8 calibration, otherwise quantization bakes in the wrong statistics."""
    has_bn = False
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            m.reset_running_stats(); m.momentum = None; has_bn = True
    if not has_bn:
        return model
    model.train()
    for _ in range(n_pass):
        for xs, xt, xw_s, xw_t, y, d in loader:
            model((xs if which == 's' else xt).to(device))
    model.eval()
    return model


def train_da_epoch(model, src_loader, tgt_loader, criterion, optimizer, device,
                   epoch, max_epoch, cfg, which='s',
                   teacher=None, kd_proj=None, ema=None,
                   pseudo_enabled=False,
                   use_coral=True, use_mmd=True, use_im=True, use_mcc=True, use_kd=True):
    """One epoch of domain-adaptive (+ optionally distilled) training.

    Key structural differences from v6:
      * source and target go through ONE concatenated forward -> clean BN statistics (FIX #5)
      * alignment losses run on at most cfg['da_passes'] * len(tgt_loader) steps (FIX #2)
      * KD uses the paired native-resolution teacher view from DualResDataset (FIX #4)
      * pseudo-labels come from the EMA model on the WEAK view, applied to the STRONG view
    """
    model.train()
    lam = ramp(epoch, max_epoch)
    w = dict(coral=cfg['coral'] * lam if use_coral else 0.0,
             mmd=cfg['mmd'] * lam if use_mmd else 0.0,
             im=cfg['im'] * lam if use_im else 0.0,
             mcc=cfg['mcc'] * lam if use_mcc else 0.0,
             pseudo=cfg['pseudo'] * lam if pseudo_enabled else 0.0,
             kd=cfg['kd'] if (use_kd and teacher is not None) else 0.0,
             featkd=cfg['featkd'] if (use_kd and teacher is not None and kd_proj is not None) else 0.0)

    n_da = min(len(src_loader), max(1, cfg['da_passes'] * len(tgt_loader)))
    tgt_iter = iter(tgt_loader)

    acc_loss = {k: 0.0 for k in ['cls', 'coral', 'mmd', 'im', 'mcc', 'pseudo', 'kd', 'featkd']}
    n_seen, n_da_seen = 0, 0
    preds, labs = [], []
    ps_used, ps_correct = 0, 0

    for step, (xs_s, xs_t, _w1, _w2, ys, _d) in enumerate(src_loader):
        do_da = step < n_da
        x_src = (xs_s if which == 's' else xs_t).to(device, non_blocking=True)
        ys = ys.to(device, non_blocking=True)
        bs = x_src.size(0)

        if do_da:
            try:
                xt_s, xt_t, xt_ws, xt_wt, yt_true, _ = next(tgt_iter)
            except StopIteration:
                tgt_iter = iter(tgt_loader)
                xt_s, xt_t, xt_ws, xt_wt, yt_true, _ = next(tgt_iter)
            x_tgt = (xt_s if which == 's' else xt_t).to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # ---- single concatenated forward: clean BN statistics (FIX #5) ----
        if do_da:
            feats = model.extract_features(torch.cat([x_src, x_tgt], 0))
            f_s, f_t = feats[:bs], feats[bs:]
            logits_s, logits_t = model.classifier(f_s), model.classifier(f_t)
        else:
            f_s = model.extract_features(x_src)
            f_t, logits_t = None, None
            logits_s = model.classifier(f_s)

        L_cls = criterion(logits_s, ys)
        loss = L_cls
        parts = {k: 0.0 for k in acc_loss}
        parts['cls'] = L_cls.item()

        if do_da:
            if w['coral'] > 0:
                l = coral_loss(f_s, f_t);  loss = loss + w['coral'] * l;  parts['coral'] = l.item()
            if w['mmd'] > 0:
                l = mmd_loss(f_s, f_t);    loss = loss + w['mmd'] * l;    parts['mmd'] = l.item()
            if w['im'] > 0:
                l = im_loss(logits_t);     loss = loss + w['im'] * l;     parts['im'] = l.item()
            if w['mcc'] > 0:
                l = mcc_loss(logits_t);    loss = loss + w['mcc'] * l;    parts['mcc'] = l.item()

            if w['pseudo'] > 0 and ema is not None:
                with torch.no_grad():
                    # weak view at the resolution THIS model consumes -> pseudo-labels
                    xw = (xt_ws if which == 's' else xt_wt).to(device)
                    weak_logits = ema.model(xw)
                keep, plab = balanced_pseudo(weak_logits, cfg['pseudo_topk'], cfg['pseudo_conf'])
                if keep.sum() > 0:
                    l = F.cross_entropy(logits_t[keep], plab[keep])  # strong view -> trained on
                    loss = loss + w['pseudo'] * l
                    parts['pseudo'] = l.item()
                    ps_used += int(keep.sum())
                    ps_correct += int((plab[keep].cpu() == yt_true[keep.cpu()]).sum())

        # ---- distillation on the paired native-resolution teacher view (FIX #4) ----
        if w['kd'] > 0:
            with torch.no_grad():
                tf_s = teacher.extract_features(xs_t.to(device, non_blocking=True))
                tl_s = teacher.classifier(tf_s)
                if do_da:
                    tf_t = teacher.extract_features(xt_t.to(device, non_blocking=True))
                    tl_t = teacher.classifier(tf_t)
            l = kd_loss_fn(logits_s, tl_s, cfg['T'])
            if do_da:
                l = 0.5 * (l + kd_loss_fn(logits_t, tl_t, cfg['T']))
            loss = loss + w['kd'] * l
            parts['kd'] = l.item()

            if w['featkd'] > 0:
                lf = feat_kd_loss(kd_proj(f_s), tf_s)
                if do_da:
                    lf = 0.5 * (lf + feat_kd_loss(kd_proj(f_t), tf_t))
                loss = loss + w['featkd'] * lf
                parts['featkd'] = lf.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if ema is not None:
            ema.update(model)

        for k in acc_loss:
            acc_loss[k] += parts[k] * bs
        n_seen += bs
        n_da_seen += bs if do_da else 0
        preds.extend(logits_s.argmax(1).detach().cpu().numpy())
        labs.extend(ys.cpu().numpy())

    out = {f'{k}_loss': v / max(n_seen, 1) for k, v in acc_loss.items()}
    out.update(train_acc=accuracy_score(labs, preds),
               train_f1=f1_score(labs, preds, average='weighted'),
               lam=lam, pseudo_used=ps_used,
               pseudo_acc=(ps_correct / ps_used) if ps_used else float('nan'),
               weights=w)
    return out


def make_scheduler(opt, epochs, warmup):
    def fn(e):
        return (e + 1) / max(warmup, 1) if e < warmup else \
               0.5 * (1 + math.cos(math.pi * (e - warmup) / max(epochs - warmup, 1)))
    return optim.lr_scheduler.LambdaLR(opt, fn)


cls_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
print("Utilities ready: EMA, EarlyStopping, evaluate, adapt_bn, train_da_epoch, make_scheduler")

Utilities ready: EMA, EarlyStopping, evaluate, adapt_bn, train_da_epoch, make_scheduler


---
# Part 5 (replaced): Load pre-trained teacher from `teacher.pth`

In [ ]:
# ============================================================
# PART 5 (REPLACED): LOAD TEACHER FROM teacher.pth  (no training)
# ============================================================
TEACHER_PATH = f"{DRIVE_ROOT}/teacher.pth"   # upload teacher.pth to this Drive folder

if not TEACHER_PATH:
    _found = sorted(glob.glob(f"{DRIVE_ROOT}/**/teacher.pth", recursive=True))
    assert _found, "teacher.pth not found -- set TEACHER_PATH manually"
    TEACHER_PATH = _found[0]
assert os.path.exists(TEACHER_PATH), f"teacher.pth not found at {TEACHER_PATH}"
print(f"Loading teacher from: {TEACHER_PATH}")

teacher = TeacherNet().to(DEVICE)
_state = torch.load(TEACHER_PATH, map_location=DEVICE)
teacher.load_state_dict(_state)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)
t_params = sum(p.numel() for p in teacher.parameters())
print(f"Teacher: {TEACHER_BACKBONE}, {t_params:,} params loaded")


Loading teacher from: /content/drive/MyDrive/microbrainnet_ablation/teacher.pth
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 103MB/s]


Teacher: mobilenet_v3_large, 2,974,835 params loaded


---
# Part 5B: Teacher sanity + AdaBN check

In [ ]:
# ============================================================
# PART 5B: TEACHER SANITY + AdaBN CHECK
# FIX #13: AdaBN keep/discard is now decided on target-VAL, never target-TEST.
# The old version compared t_ab_acc vs t_te_acc (both computed on tgt_test_loader) and
# kept whichever scored higher ON THE TEST SET -- that's test-set peeking: reporting the
# best-of-2 test score instead of a single clean held-out measurement. Now the decision
# uses tgt_val_loader, and tgt_test_loader is touched exactly once, with the model the
# decision already settled on.
# ============================================================
if USE_TEACHER_KD:
    _, t_val_acc, _, _, _, _ = evaluate(teacher, tgt_val_loader, cls_criterion, DEVICE, 't')

    teacher_adabn = copy.deepcopy(teacher)
    adapt_bn(teacher_adabn, tgt_train_loader, DEVICE, which='t', n_pass=3)
    _, t_ab_val_acc, _, _, _, _ = evaluate(teacher_adabn, tgt_val_loader, cls_criterion, DEVICE, 't')

    print(f"Teacher target-VAL (no AdaBN) : acc={t_val_acc:.4f}")
    print(f"Teacher target-VAL (+ AdaBN)  : acc={t_ab_val_acc:.4f} (delta {t_ab_val_acc - t_val_acc:+.4f})")

    if t_ab_val_acc >= t_val_acc:
        teacher = teacher_adabn
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
        print("  -> AdaBN version selected (decided on target-VAL, not test).")
    else:
        del teacher_adabn
        print("  -> Base (non-AdaBN) version kept (decided on target-VAL, not test).")
    torch.cuda.empty_cache()

    # Test set touched exactly once, with whichever model the VAL-based decision above settled on.
    _, t_te_acc, t_te_f1, t_pred, t_lab, _ = evaluate(teacher, tgt_test_loader,
                                                      cls_criterion, DEVICE, 't')
    print(f"\nTeacher target-TEST (final, single measurement) : acc={t_te_acc:.4f} f1={t_te_f1:.4f}")
    print("Teacher target-test confusion (rows = actual):")
    cm = confusion_matrix(t_lab, t_pred)
    print(" " * 27 + "".join(f"{c[:13]:>15s}" for c in CLASSES))
    for i, c in enumerate(CLASSES):
        print(f"{c:<27s}" + "".join(f"{cm[i][j]:>15d}" for j in range(NUM_CLASSES)))


Teacher target-VAL (no AdaBN) : acc=0.8228
Teacher target-VAL (+ AdaBN)  : acc=0.8354 (delta +0.0127)
  -> AdaBN version selected (decided on target-VAL, not test).

Teacher target-TEST (final, single measurement) : acc=0.7848 f1=0.7893
Teacher target-test confusion (rows = actual):
                             Tomato_Septor  Potato_Early_  Corn_Leaf_Bli
Tomato_Septoria_Leaf_Spot               22              2              0
Potato_Early_Blight                     10             16              0
Corn_Leaf_Blight                         4              1             24


---
# Helper: bootstrap CI (Part 6 theke, ablation e lage)

In [ ]:
def bootstrap_ci(labels, preds, n_boot=5000, seed=SEED):
    rng = np.random.RandomState(seed)
    labels, preds = np.asarray(labels), np.asarray(preds)
    n = len(labels)
    accs = np.array([(labels[i] == preds[i]).mean()
                     for i in (rng.randint(0, n, n) for _ in range(n_boot))])
    return accs.mean(), *np.percentile(accs, [2.5, 97.5])


---
# Part 9: Ablation

All rows share one budget (`ABL_EPOCHS`, seed 42, same teacher, same loaders), so they are comparable
with each other. `ABL_EPOCHS = 120`. Even so, the ablation loop differs from the main run in Part 5C/6 (no early stopping, warmup 3, fixed pseudo-label start, AdaBN always applied), so ablation numbers should be compared with each other, not with the main result.

This run is **IM-only** (`USE_CORAL=USE_MMD=USE_MCC=False`, Part 0), so `Full` means
"IM + pseudo-labels + KD + AdaBN". Each row removes exactly one component from `Full`:

| Row | What is removed |
|---|---|
| `w/o KD` | logit + feature distillation from the teacher |
| `w/o IM (Domain Adaptation)` | the Information-Maximization alignment loss -- IM is the *only* alignment term in this run, so this row **is** the "no domain adaptation" row |
| `w/o Pseudo-Labeling` | EMA / class-balanced self-training |
| `w/o AdaBN` | target-only BatchNorm re-estimation before evaluation (the model keeps the BN statistics from mixed-domain training) |
| `w/o SE` | Squeeze-Excite block |
| `w/o Micro-ViT` | linear-attention block (global average pool instead) |

The old separate `w/o Domain Adaptation` row was dropped: with IM as the only alignment loss it
duplicated `w/o IM`.

On 79 target-test images, rows whose 95% bootstrap CIs overlap are not statistically separable.


In [ ]:
# ============================================================
# PART 9: ABLATION
# ============================================================
ABL_EPOCHS = 120   # all ablation rows trained for 120 epochs
ABL_SEED   = 42    # ablation runs on a single seed (42) only -- no multi-seed needed

# IM-only run: CORAL/MMD/MCC stay at USE_CORAL=USE_MMD=USE_MCC=False (Part 0) throughout.
ablation = OrderedDict([
    # name                          (se,    vit,   coral,     mmd,     im,      mcc,     pseudo, kd,    adabn)
    ('Full (IM-only DA)',           (True,  True,  USE_CORAL, USE_MMD, USE_IM,  USE_MCC, True,   True,  True)),
    ('w/o KD',                      (True,  True,  USE_CORAL, USE_MMD, USE_IM,  USE_MCC, True,   False, True)),
    ('w/o IM (Domain Adaptation)',  (True,  True,  USE_CORAL, USE_MMD, False,   USE_MCC, True,   True,  True)),
    ('w/o Pseudo-Labeling',         (True,  True,  USE_CORAL, USE_MMD, USE_IM,  USE_MCC, False,  True,  True)),
    ('w/o AdaBN',                   (True,  True,  USE_CORAL, USE_MMD, USE_IM,  USE_MCC, True,   True,  False)),
    ('w/o SE',                      (False, True,  USE_CORAL, USE_MMD, USE_IM,  USE_MCC, True,   True,  True)),
    ('w/o Micro-ViT',               (True,  False, USE_CORAL, USE_MMD, USE_IM,  USE_MCC, True,   True,  True)),
])


# ------------------------------------------------------------
# COLAB SPLIT RUN: choose which rows to run in THIS session.
# Each row already calls set_seed(ABL_SEED) itself, so rows are independent and can run in
# different sessions. Finished rows are saved to Drive and skipped automatically next time.
# ------------------------------------------------------------
RUN_ROWS = ['w/o KD']       # e.g. RUN_ROWS = ['Full (IM-only DA)', 'w/o KD']
ABL_SAVE_DIR = f"{TABLES_DIR}/ablation_rows"
os.makedirs(ABL_SAVE_DIR, exist_ok=True)

def _row_file(name):
    # epoch count is part of the filename, so rows from a different ABL_EPOCHS are never mixed in
    return (f"{ABL_SAVE_DIR}/" + "".join(ch if ch.isalnum() else "_" for ch in name)
            + f"_ep{ABL_EPOCHS}.json")

for _n in ablation:
    print(f"  [{'DONE' if os.path.exists(_row_file(_n)) else 'todo'}] {_n}")


  [todo] Full (IM-only DA)
  [todo] w/o KD
  [todo] w/o IM (Domain Adaptation)
  [todo] w/o Pseudo-Labeling
  [todo] w/o AdaBN
  [todo] w/o SE
  [todo] w/o Micro-ViT


In [ ]:
abl_results = OrderedDict()
for abl_name, (u_se, u_vit, u_co, u_mm, u_im, u_mc, u_ps, u_kd, u_ab) in ablation.items():
    if abl_name not in RUN_ROWS:
        continue
    if os.path.exists(_row_file(abl_name)):
        print(f"skip (already saved): {abl_name}")
        continue
    set_seed(ABL_SEED)
    print(f"\n{'-'*70}\nAblation: {abl_name}")
    abl_m = MicroBrainNetDA(use_se=u_se, use_vit=u_vit).to(DEVICE)
    abl_proj = (nn.Linear(abl_m.feature_dim, teacher.feature_dim).to(DEVICE)
                if (u_kd and USE_TEACHER_KD and teacher is not None) else None)
    abl_params = list(abl_m.parameters()) + (list(abl_proj.parameters()) if abl_proj else [])
    abl_opt = optim.AdamW(abl_params, lr=LR, weight_decay=WEIGHT_DECAY)
    abl_sched = make_scheduler(abl_opt, ABL_EPOCHS, 3)
    abl_ema = EMA(abl_m) if u_ps else None
    abl_best, abl_best_state, abl_raw, abl_ps_on = 0.0, copy.deepcopy(abl_m.state_dict()), [], False

    for ep_ in range(1, ABL_EPOCHS + 1):
        st_ = train_da_epoch(abl_m, src_train_loader, tgt_train_loader, cls_criterion, abl_opt, DEVICE,
                             ep_, ABL_EPOCHS, CFG, which='s',
                             teacher=teacher if (u_kd and USE_TEACHER_KD) else None,
                             kd_proj=abl_proj, ema=abl_ema, pseudo_enabled=abl_ps_on,
                             use_coral=u_co, use_mmd=u_mm, use_im=u_im, use_mcc=u_mc, use_kd=u_kd)
        abl_sched.step()
        _, tv_, _, _, _, _ = evaluate(abl_m, tgt_val_loader, cls_criterion, DEVICE)
        abl_raw.append(tv_)
        sm_ = float(np.mean(abl_raw[-SMOOTH_WINDOW:]))
        if sm_ > abl_best:
            abl_best, abl_best_state = sm_, copy.deepcopy(abl_m.state_dict())
        if u_ps and ep_ >= ABL_EPOCHS // 3:
            abl_ps_on = True
            if st_['pseudo_used'] > 0 and st_['pseudo_acc'] < PSEUDO_MIN_MONITOR_ACC:
                abl_ps_on = False
        if ep_ % 15 == 0:
            print(f"  ep {ep_:3d}: tgt-val={tv_:.4f} (sm={sm_:.4f})")

    abl_m.load_state_dict(abl_best_state)
    if u_ab:                                    # AdaBN: re-estimate BN stats on TARGET train only
        adapt_bn(abl_m, tgt_train_loader, DEVICE, 's', 2)
    abl_m.eval()
    _, sa_, _, _, _, _ = evaluate(abl_m, src_val_loader, cls_criterion, DEVICE)
    _, ta_, tfa_, tp_, tl_, _ = evaluate(abl_m, tgt_test_loader, cls_criterion, DEVICE)
    _, clo_, chi_ = bootstrap_ci(tl_, tp_, 2000)
    abl_results[abl_name] = {'params': sum(p.numel() for p in abl_m.parameters()),
                             'src_acc': sa_, 'tgt_acc': ta_, 'tgt_f1': tfa_,
                             'ci_lo': clo_, 'ci_hi': chi_}
    print(f"  -> src={sa_:.4f} tgt={ta_:.4f} [{clo_:.3f},{chi_:.3f}] f1={tfa_:.4f}")
    del abl_m, abl_opt, abl_proj, abl_ema
    torch.cuda.empty_cache()

    # COLAB: save this row immediately so a disconnect does not lose it
    with open(_row_file(abl_name), "w") as _f:
        json.dump({k: float(v) for k, v in abl_results[abl_name].items()} |
                  {"gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
                   "torch": torch.__version__, "abl_epochs": ABL_EPOCHS, "seed": ABL_SEED}, _f, indent=1)
    print(f"  saved -> {_row_file(abl_name)}")



----------------------------------------------------------------------
Ablation: w/o KD
  ep  15: tgt-val=0.4684 (sm=0.4582)
  ep  30: tgt-val=0.5443 (sm=0.5848)
  ep  45: tgt-val=0.5949 (sm=0.6278)
  ep  60: tgt-val=0.6456 (sm=0.6329)
  ep  75: tgt-val=0.6456 (sm=0.6405)
  ep  90: tgt-val=0.5949 (sm=0.6177)
  ep 105: tgt-val=0.6456 (sm=0.6582)
  ep 120: tgt-val=0.6203 (sm=0.6228)
  -> src=0.9770 tgt=0.5696 [0.468,0.671] f1=0.5827
  saved -> /content/drive/MyDrive/microbrainnet_ablation/microbrainnet_v7/tables/ablation_rows/w_o_KD_ep120.json
